## 第18章　大数定律与中心极限定理 —— AI的"定心丸"

> 本章目标：理解两大定理——大数定律（样本均值收敛到期望）和中心极限定理（均值的分布趋向正态）。亲手验证"从任何分布重复抽样，样本均值的分布一定趋近正态"，并建立全书最重要的训练直觉：**Mini-Batch SGD 的梯度估计凭什么靠谱？因为 CLT 保证了它是对真实梯度的无偏且近似正态的估计。**
> 前置知识：第 15 章（概率分布）、第 17 章（期望与方差）、第 12 章（梯度下降）



In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import math
print("env ready")


### 18.1　大数定律 —— "试多了就准了"

掷一次硬币，正面概率 0.5——但你掷 10 次可能只有 3 次正面（频率 0.3）。掷 100 次可能是 47 次正面（频率 0.47）。掷 10000 次——频率几乎恰好是 0.5。

这不是巧合。**大数定律（Law of Large Numbers）** 严格保证了这个直觉：当试验次数 n → ∞ 时，样本均值以概率 1 收敛到真实期望。在第 15.1 节你看到骰子频率从 30 次时的参差不齐收敛到 3000 次时的几乎恰好 1/6——那就是大数定律在起作用。

📐 **定义　大数定律（LLN）**：x̄ₙ = (1/n) Σ xᵢ → E[X] 当 n → ∞。样本量越大，样本均值越接近真实期望。这是所有统计估计的根基——如果没有大数定律，"从数据中学习"这件事在数学上就不可能。

💻 **代码　硬币频率收敛 + 多条轨迹对比**


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

np.random.seed(42)

n_max = 5000
n_trajectories = 5  # 5 条独立的收敛轨迹

fig, ax = plt.subplots(figsize=(10, 5))

for traj in range(n_trajectories):
    flips = np.random.random(n_max) < 0.5  # True=正面
    cum_heads = np.cumsum(flips)
    freq = cum_heads / np.arange(1, n_max + 1)
    ax.plot(range(1, n_max + 1), freq, linewidth=1, alpha=0.7,
            label=f'试验 {traj+1}' if traj < 3 else '')

ax.axhline(y=0.5, color='red', linestyle='--', linewidth=2, label='真实概率 0.5')
# 标注 ±1/√n 的理论收敛带
n_range = np.arange(1, n_max + 1)
ax.fill_between(n_range, 0.5 - 1/np.sqrt(n_range), 0.5 + 1/np.sqrt(n_range),
                alpha=0.1, color='gray', label='理论 ±1/√n')

ax.set_xlabel('试验次数 n'); ax.set_ylabel('正面频率')
ax.set_title('大数定律：频率随 n 增大收敛到 0.5')
ax.legend(fontsize=8); ax.grid(alpha=0.3); plt.show()

print("观察: n<100 时频率剧烈波动(偶然性主导)")
print("      n≈1000 时频率稳定在 0.48~0.52")
print("      n→∞ 时频率→0.5(以概率1)")


> **关键洞察**：大数定律是 AI 的"定心丸"——它告诉你**只要数据足够多，统计估计就一定准。** 训练集中 loss 的下降、验证集上准确率的稳定、A/B 测试中点击率的可信度——全都在靠大数定律"兜底"。反之，如果只有 10 个样本就声称"准确率 90%"，大数定律会警告你：样本太少，频率不可信。

🔗 **AI 连接**：第 24 章 Mini-Batch SGD 中，batch_size 不能太小——因为太小的 batch 违反了"大数"的前提，梯度估计方差太大，SGD 收敛不稳定。

---


### 18.2　中心极限定理 —— 任何分布，均值都趋向正态

大数定律告诉你"均值最终会准"。但**中心极限定理（Central Limit Theorem, CLT）** 回答了一个更深刻的问题：**在收敛过程中，均值有多分散？它的分布长什么样？**

答案令人震惊：**不管你从什么分布抽样（偏态、均匀、甚至双峰），只要样本量 n 足够大，样本均值的分布一定趋向正态分布 N(μ, σ²/n)。**

这不是近似——这是一个数学定理。均匀分布是方的，指数分布是偏的，二项分布是离散的——但从这些分布中每次抽 30 个样本算均值，重复 10000 次，10000 个均值的直方图一定是一口漂亮的钟形曲线。

📐 **定义　中心极限定理（CLT）**：若 x₁, ..., xₙ i.i.d. 来自均值为 μ、方差为 σ² 的任意分布，则当 n → ∞ 时，x̄ₙ ~ N(μ, σ²/n)。**均值的方差是原始方差的 1/n**——这就是为什么大样本下的估计更精确。

💻 **代码　从指数分布（极度偏态！）抽样，亲眼见证均值趋向正态**


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from math import erf

def normal_cdf(x):
    return 0.5 * (1 + erf(x / np.sqrt(2)))

np.random.seed(42)

# 指数分布——极度右偏（PDF：λe^{-λx}, x>0），和正态分布天差地别
lam = 1.0
pop_mean = 1 / lam    # 指数分布均值 = 1/λ = 1
pop_std = 1 / lam     # 指数分布标准差 = 1/λ = 1

n_exp = 10000  # 重复实验次数

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# --- 左：原始指数分布（极度偏态）---
samples_raw = np.random.exponential(1/lam, size=5000)
axes[0].hist(samples_raw, bins=50, density=True, color='steelblue', edgecolor='white')
axes[0].set_title(f'原始分布: 指数分布(λ={lam})\n极度右偏，和正态完全不像')
axes[0].set_xlabel('x'); axes[0].set_ylabel('密度')

# --- 中、右：不同样本量下的均值分布 ---
for idx, (n, color) in enumerate(zip([5, 30], ['coral', 'steelblue'])):
    # 每次抽 n 个样本，算均值，重复 n_exp 次
    sample_means = np.random.exponential(1/lam, size=(n_exp, n)).mean(axis=1)
    # 理论: 均值的分布 ~ N(pop_mean, pop_std²/n)
    theo_std = pop_std / np.sqrt(n)

    ax = axes[idx + 1]
    ax.hist(sample_means, bins=60, density=True, alpha=0.7, color=color, edgecolor='white')
    x_grid = np.linspace(pop_mean - 4*theo_std, pop_mean + 4*theo_std, 200)
    # 理论正态 PDF
    pdf = np.exp(-0.5 * ((x_grid - pop_mean) / theo_std)**2) / (theo_std * np.sqrt(2*np.pi))
    ax.plot(x_grid, pdf, 'r-', linewidth=2, label=f'N({pop_mean}, {pop_std**2}/{n})')

    # 验证 68-95 规则
    in_1sig = np.mean(np.abs(sample_means - pop_mean) < theo_std)
    in_2sig = np.mean(np.abs(sample_means - pop_mean) < 2*theo_std)
    ax.set_title(f'n={n} 个样本取均值\n±1σ内: {in_1sig:.1%}  ±2σ内: {in_2sig:.1%}')
    ax.set_xlabel('样本均值'); ax.set_ylabel('密度')
    ax.legend(fontsize=8)

plt.suptitle('中心极限定理：指数分布(偏态)→均值分布→正态！', fontsize=13, y=1.02)
plt.tight_layout(); plt.show()

print(f"原始指数分布: 均值={pop_mean}, 标准差={pop_std}  (极度偏态!)")
print(f"n=5 时均值的标准差={pop_std/np.sqrt(5):.3f}  n=30时={pop_std/np.sqrt(30):.3f}")
print("CLT 结论: 不管原始分布长什么样, 均值的分布一定趋向正态")


> **关键洞察**：n=5 时均值分布已呈现钟形（虽然还有点偏），n=30 时几乎完美正态——而且均值的离散度（标准差）从原始的 1.0 缩小到 1/√30 ≈ 0.18。**这就是为什么增大 batch size 能让梯度估计更稳定：CLT 保证了均值的方差 ∝ 1/n。**

🔗 **AI 连接**：第 24 章 Mini-Batch SGD 中，每一步的梯度是对 batch 内各样本梯度的平均——CLT 保证了这个小批量平均梯度是大批量真实梯度的无偏且近似正态的估计。这就是为什么 SGD 能用小批量替代全量：**CLT 是大规模分布式训练可以收敛的数学保证。**

---


### 18.3　AI 核心：Mini-Batch SGD 凭什么靠谱？

现在把 CLT 直接应用到深度学习训练：

全量梯度（所有样本的平均）= `∇L_full = (1/N) Σᵢ ∇L(xᵢ)`。Mini-Batch 梯度（随机抽 B 个样本）= `∇L_batch = (1/B) Σⱼ ∇L(xⱼ)`。

CLT 告诉你：**当 batch size B 足够大时，∇L_batch 的分布近似 N(∇L_full, σ²/B)。** 这意味着：

1. **无偏性**：Mini-Batch 梯度"平均而言"指向全量梯度方向（E[∇L_batch] = ∇L_full）
2. **方差可控**：梯度的噪声水平 σ²/B 随 batch size 增大而减小——想更稳定就增大 B
3. **渐近正态**：B 足够大后，梯度的误差服从正态分布——可以用置信区间理论指导调参

**这就是 Mini-Batch SGD 能工作的数学根基：CLT 保证了随机采样带来的梯度噪声是可控的、正态的、且随 batch size 增大而衰减的。** 没有这个保证，每次参数更新都是"走一步不知道会偏到哪去"的赌博——有了 CLT，你才能自信地让训练跑上几百万步。

📐 **核心公式**：`∇L_batch ~ N(∇L_full, σ²/B)`。B=32 时的梯度噪声是 B=128 时的 √(128/32) = 2 倍。工业界常用 B=32~256 作为"甜点区间"——够大保证 CLT 收敛，够小保证单步计算快。

💻 **代码　验证 CLT 对 SGD 梯度的意义：不同 batch size 的梯度方差对比**


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

np.random.seed(42)

# 模拟：10000 个样本的梯度（来自某个偏态分布——真实场景梯度的分布往往偏态）
N_total = 10000
all_grads = np.random.exponential(0.5, size=N_total) - 0.25  # 偏态，均值为 0.25
true_grad = all_grads.mean()

print(f"全量梯度(真实值): {true_grad:.4f}")
print(f"梯度总体标准差: {all_grads.std():.3f}\n")

batch_sizes = [8, 32, 128, 512]
n_trials = 2000

fig, axes = plt.subplots(1, len(batch_sizes), figsize=(14, 3.5))

for idx, B in enumerate(batch_sizes):
    batch_means = np.array([
        np.random.choice(all_grads, size=B, replace=True).mean()
        for _ in range(n_trials)
    ])

    theo_std = all_grads.std() / np.sqrt(B)
    actual_std = batch_means.std()

    axes[idx].hist(batch_means, bins=40, density=True, alpha=0.7,
                   color='steelblue', edgecolor='white')
    axes[idx].axvline(x=true_grad, color='red', linestyle='--', linewidth=2, label=f'True grad')
    axes[idx].set_title(f'B={B}\ntheo σ={theo_std:.4f}\nactual σ={actual_std:.4f}')
    axes[idx].set_xlabel('Batch gradient'); axes[idx].legend(fontsize=7)

plt.suptitle('CLT 验证：batch size 越大 → 梯度估计越集中 → SGD 越稳定', fontsize=13)
plt.tight_layout(); plt.show()

print(f"{'B':<6} {'理论σ':<12} {'实际σ':<12} {'B=512相对噪声'}")
for idx, B in enumerate(batch_sizes):
    theo = all_grads.std() / np.sqrt(B)
    actual = batch_means.std()
    relative = actual / true_grad if abs(true_grad) > 1e-6 else float('inf')
    print(f"{B:<6} {theo:<12.4f} {actual:<12.4f} {relative:.2%}")


> **关键洞察**：B=8 时梯度估计的分布非常分散（方差大）——SGD 步的方向不稳定，可能"走歪"。B=512 时梯度估计紧密聚集在真实梯度周围（方差小）——SGD 步的方向可靠。**但 batch size 不是越大越好**：B=512 计算一次梯度的时间是 B=8 的 64 倍，而方差只缩小了 8 倍（∝1/√B，不是 1/B）。工业界在 B=32~256 之间选择，是在"计算效率"和"梯度精度"之间的最优平衡。第 24 章将讨论 Gradient Accumulation 技术——用多次小 batch 的前向+反向累积大 batch 的梯度效果。

🔗 **AI 连接**：第 24 章优化器（Momentum、Adam）在设计时假设了"梯度是带噪声的真实梯度估计"——CLT 正是这个假设的数学依据。第 31 章训练 GPT-2 时，`per_device_train_batch_size=8` 和 `gradient_accumulation_steps=4` 的组合就是在利用 CLT：用小 batch 累积等效大 batch 的梯度精度。

---


**✏️ 习题**

> 在下方新建代码单元格作答。
1. （概念）大数定律和中心极限定理的核心区别是什么？各用一句话概括。
2. （概念）为什么 Mini-Batch SGD 中的梯度估计是合理的？CLT 是怎么保证这一点的？如果 batch_size=1 会怎样？
3. （代码）从均匀分布 Uniform(0, 10) 抽样，分别用 n=2, 10, 50 取样本均值，重复 5000 次。画出三个直方图对比，验证均值的分布随 n 增大越来越接近正态，且方差越来越小。
---
> 🔗 **章末钩子**：大数定律和 CLT 告诉你"数据多了就准、分布趋向正态"。但它们没回答一个更根本的问题：损失函数为什么要设计成"交叉熵"这种形式？"熵"到底是什么？为什么它能衡量"不确定性"？
> 预览：下一章——**信息论**，损失函数的终极来源。从熵到交叉熵到 KL 散度，理解为什么你的 loss function 长成这样。
